# 核函数定义与JIT编译

## 概述

本节深入讲解pyasc核函数的定义方法、参数类型体系、JIT编译缓存机制，以及如何查看编译生成的中间代码。核函数是算子设备侧的入口，理解其定义规范和编译机制是编写正确pyasc算子的前提。

### 学习目标

1. 掌握核函数的参数定义方法（GlobalAddress、int、ConstExpr等）；
2. 理解编译时参数与运行时参数的区别；
3. 掌握JIT编译缓存机制和环境变量配置；
4. 学会通过`PYASC_DUMP_PATH`查看编译生成的ASC-IR和Ascend C代码。


In [ ]:
# 环境初始化
!mkdir -p Sources/02.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. 核函数定义

核函数（Kernel Function）是pyasc算子设备侧的入口，也是Host侧和Device侧连接的桥梁。在Ascend C中，核函数通过`__global__ __aicore__`修饰符定义；在pyasc中，使用`@asc.jit`装饰器替代。

### 1.1 Add算子核函数签名

以Add算子为例，其核函数定义如下：

```python
@asc.jit
def vadd_kernel(x: asc.GlobalAddress, y: asc.GlobalAddress, z: asc.GlobalAddress, block_length: int):
    offset = asc.get_block_idx() * block_length
    ...
```

### 1.2 核函数参数类型

pyasc核函数的参数类型与Ascend C一一对应，但使用Python类型注解语法：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">参数类型</th>
      <th align="left">说明</th>
      <th align="left">对应Ascend C</th>
      <th align="left">分类</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>asc.GlobalAddress</code></td><td>Global Memory地址</td><td><code>__gm__ T*</code></td><td>运行时参数</td></tr>
    <tr><td><code>int</code></td><td>整数</td><td><code>int32_t</code></td><td>运行时参数</td></tr>
    <tr><td><code>float</code></td><td>浮点数</td><td><code>float</code></td><td>运行时参数</td></tr>
    <tr><td><code>bool</code></td><td>布尔值</td><td><code>bool</code></td><td>运行时参数</td></tr>
    <tr><td><code>asc.ConstExpr[int]</code></td><td>编译时常量</td><td>模板参数</td><td>编译时参数</td></tr>
  </tbody>
</table>

### 1.3 ConstExpr编译时常量

`asc.ConstExpr[int]`表示参数在**编译时**确定值，在生成的Ascend C代码中作为模板参数生效。与运行时参数的区别：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">类型</th>
      <th align="left">声明方式</th>
      <th align="left">值确定时机</th>
      <th align="left">编译优化</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>运行时参数</td><td><code>block_length: int</code></td><td>运行时</td><td>不可优化</td></tr>
    <tr><td>编译时常量</td><td><code>BUFFER_NUM: asc.ConstExpr[int]</code></td><td>编译时</td><td>编译器可优化</td></tr>
  </tbody>
</table>

ConstExpr适用于缓冲区数量（BUFFER_NUM）、Tile长度（TILE_LENGTH）等编译时就能确定的常量。

### 1.4 ConstExpr使用示例

在框架版Add样例（`add_framework.py`）中，`tile_length`被声明为ConstExpr：

```python
# add_framework.py 中的核函数签名
@asc.jit
def vadd_kernel(x: asc.GlobalAddress, y: asc.GlobalAddress, z: asc.GlobalAddress,
                block_length: int,                          # 运行时参数
                tile_length: asc.ConstExpr[int]):            # 编译时常量
    ...
```

对比手动版Add样例（`add.py`），`tile_length`在核函数内部计算，是运行时变量：

```python
# add.py 中的核函数签名
@asc.jit
def vadd_kernel(x: asc.GlobalAddress, y: asc.GlobalAddress, z: asc.GlobalAddress,
                block_length: int):                          # 全部是运行时参数
    tile_length = block_length // TILE_NUM // BUFFER_NUM     # 运行时计算
    ...
```

ConstExpr方式让编译器在编译时就知道tile_length的值，可以进行更好的优化。


---
# 2. JIT编译流程

pyasc的`@asc.jit`装饰器会触发JIT（Just-In-Time）编译流程，将Python函数逐步转换为NPU可执行的Kernel二进制：

```text
Python源码 → AST → ASC-IR (MLIR) → Ascend C代码 → 毕昇编译器 → Kernel二进制
```

各阶段说明：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">阶段</th>
      <th align="left">说明</th>
      <th align="left">所属模块</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Python源码</td><td>开发者编写的<code>@asc.jit</code>修饰函数</td><td>前端</td></tr>
    <tr><td>AST</td><td>Python抽象语法树，由编译模块解析</td><td>前端</td></tr>
    <tr><td>ASC-IR</td><td>基于MLIR的中间表示，1:1映射Ascend C API</td><td>前端→后端</td></tr>
    <tr><td>Ascend C代码</td><td>由ASC-IR生成的C++代码</td><td>后端</td></tr>
    <tr><td>Kernel二进制</td><td>毕昇编译器编译后的NPU可执行文件</td><td>后端</td></tr>
  </tbody>
</table>

## 2.1 JIT编译缓存机制

pyasc提供**JIT编译缓存**功能，能够存储和复用已经编译的Kernel二进制，避免重复编译，从而提高编译效率和程序执行速度。

缓存的工作方式是：首次调用某个核函数时，pyasc会执行完整的编译流程并缓存结果；后续调用相同核函数时（编译选项、参数、代码均未变化），直接从缓存加载Kernel二进制，跳过编译过程。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">环境变量</th>
      <th align="left">说明</th>
      <th align="left">默认值</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>PYASC_HOME</code></td><td>缓存根目录</td><td>当前用户目录</td></tr>
    <tr><td><code>PYASC_CACHE_DIR</code></td><td>具体缓存目录</td><td><code>${PYASC_HOME}/.pyasc/cache</code></td></tr>
    <tr><td><code>PYASC_DUMP_PATH</code></td><td>编译产物保存路径</td><td>不保存</td></tr>
  </tbody>
</table>

缓存影响因素：编译选项、Kernel参数、全局变量、Kernel函数代码。如果需要强制重新编译，设置`always_compile=True`：

```python
@asc.jit(always_compile=True)
def kernel_func(*params):
    ...
```


---
# 3. 查看编译生成的代码

通过设置`PYASC_DUMP_PATH`环境变量，可以保存编译过程中生成的ASC-IR和Ascend C代码，方便开发者理解pyasc的编译过程。

下面我们设置dump路径并运行Add样例，查看编译生成的中间文件：

In [ ]:
# 设置dump路径并运行Add样例，查看编译生成的文件
import os
os.environ['PYASC_DUMP_PATH'] = '/tmp/pyasc_dump'

!PYASC_DUMP_PATH=/tmp/pyasc_dump python3 ./src/add.py -r NPU

In [ ]:
# 查看dump目录下生成的文件
!find /tmp/pyasc_dump -type f 2>/dev/null | head -20

### 3.1 编译产物说明

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">文件类型</th>
      <th align="left">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>codegen.mlir</code></td><td>Python AST转ASC-IR的中间表示</td></tr>
    <tr><td><code>ascendc.cpp</code></td><td>最终生成的Ascend C代码</td></tr>
    <tr><td>编译日志</td><td>毕昇编译器的编译过程日志</td></tr>
  </tbody>
</table>


In [ ]:
# 查看生成的ASC-IR中间表示（截取前40行）
!find /tmp/pyasc_dump -name 'codegen.mlir' -exec head -40 {} \;

### 3.2 pyasc源码与生成Ascend C代码对照

通过对比pyasc源码和生成的Ascend C代码，可以直观理解pyasc的接口映射关系：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">pyasc源码</th>
      <th align="left">生成的Ascend C代码</th>
      <th align="left">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>@asc.jit</code></td><td><code>__global__ __aicore__</code></td><td>核函数修饰符</td></tr>
    <tr><td><code>asc.GlobalAddress</code></td><td><code>__gm__ uint8_t*</code></td><td>GM地址参数</td></tr>
    <tr><td><code>asc.GlobalTensor()</code></td><td><code>GlobalTensor&lt;T&gt;</code></td><td>全局Tensor</td></tr>
    <tr><td><code>asc.data_copy(...)</code></td><td><code>DataCopy(...)</code></td><td>数据搬运</td></tr>
    <tr><td><code>asc.add(...)</code></td><td><code>Add(...)</code></td><td>矢量加法</td></tr>
    <tr><td><code>asc.set_flag(...)</code></td><td><code>SetFlag(...)</code></td><td>设置同步事件</td></tr>
    <tr><td><code>asc.wait_flag(...)</code></td><td><code>WaitFlag(...)</code></td><td>等待同步事件</td></tr>
  </tbody>
</table>


In [ ]:
# 查看生成的Ascend C代码（截取前60行）
!find /tmp/pyasc_dump -name 'ascendc.cpp' -exec head -60 {} \;

可以看到，pyasc生成的Ascend C代码与你手写Ascend C代码结构类似，包含`__global__ __aicore__`核函数定义、`GlobalTensor`/`LocalTensor`对象、`DataCopy`/`Add`/`SetFlag`/`WaitFlag`等接口调用。这验证了pyasc与Ascend C接口的一一对应关系。

---
# 4. 小结

- `@asc.jit`装饰器替代了Ascend C的`__global__ __aicore__`修饰符
- 核函数参数支持`asc.GlobalAddress`、`int`、`float`、`asc.ConstExpr`等类型
- `asc.ConstExpr[int]`表示编译时常量，对应Ascend C模板参数
- JIT编译缓存机制避免重复编译，`always_compile=True`可强制重编译
- 通过`PYASC_DUMP_PATH`可查看ASC-IR和Ascend C代码等编译产物


---

## 课后练习

**选择题：**

1. `asc.GlobalAddress`参数类型对应Ascend C中的什么？
   - A. 局部变量
   - B. Global Memory指针
   - C. 常量
   - D. 模板参数

2. `asc.ConstExpr[int]`参数在什么时候确定值？
   - A. 运行时
   - B. 编译时
   - C. 调试时
   - D. 安装时

**填空题：**

3. 设置环境变量________可以指定编译生成文件的保存路径。

4. 如果需要强制重新编译忽略缓存，应设置编译参数________=True。

<details>
<summary>点击查看答案</summary>

1. B  2. B  3. `PYASC_DUMP_PATH`  4. `always_compile`

</details>